<a href="https://colab.research.google.com/github/PujaSridhar/finsight/blob/main/notebooks/03_evaluate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FinSight · Phase 3: base vs. fine-tuned evaluation

Runs the base model and the fine-tuned adapter on three held-out sets with the **same prompt**:
FinQA test (300), TAT-QA dev (300) and FinanceBench (150, never used in training).
Each model takes about 45–75 minutes on a T4. Predictions are saved as they go, so re-running resumes.

In [ ]:
# Clone the repo and install dependencies (~2 min)
REPO_BRANCH = "main"
!git clone -q -b $REPO_BRANCH https://github.com/PujaSridhar/finsight /content/finsight 2>/dev/null || (cd /content/finsight && git pull -q)
%cd /content/finsight
!pip install -q -r requirements.txt

In [ ]:
# Keep checkpoints and results on Google Drive so a disconnect doesn't lose them.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/finsight/outputs /content/drive/MyDrive/finsight/results
!rm -rf outputs results && ln -s /content/drive/MyDrive/finsight/outputs outputs && ln -s /content/drive/MyDrive/finsight/results results

In [ ]:
# Mistral-7B-Instruct-v0.3 is gated: accept its terms at
# https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3, then paste a read token here
# (or store it as the Colab secret HF_TOKEN).
from huggingface_hub import login
login()

In [ ]:
!python -m finsight.data --out data/processed

In [ ]:
!python -m finsight.evaluate --tag base

In [ ]:
!python -m finsight.evaluate --tag finetuned --adapter outputs/finsight-mistral-7b-qlora/final

## Optional: LLM judge for FinanceBench's free-text answers
Needs an Anthropic API key (a few cents for 300 calls). Skip it otherwise.

In [ ]:
# import os; os.environ["ANTHROPIC_API_KEY"] = "..."   # or use a Colab secret
# !pip install -q anthropic
# !python -m finsight.judge results/predictions/base_financebench.jsonl results/predictions/finetuned_financebench.jsonl
# !python -m finsight.evaluate --tag base --sets financebench && python -m finsight.evaluate --tag finetuned --sets financebench --adapter outputs/finsight-mistral-7b-qlora/final

## Results table (copy into the README)

In [ ]:
!python -m finsight.evaluate --report
from IPython.display import Markdown, display
display(Markdown(open("results/RESULTS.md").read()))

## Error analysis: where does the fine-tuned model still fail?

In [ ]:
import json, collections
preds = [json.loads(l) for l in open("results/predictions/finetuned_finqa.jsonl")]
wrong = [p for p in preds if not p["correct"]]
print(f"{len(wrong)} wrong out of {len(preds)}")
for p in wrong[:8]:
    print("Q:", p["question"], "\nGOLD:", p["answer"], "\nMODEL:", p["response"], "\n" + "-" * 60)